In [0]:
from lakehouse.transforms.macro_builder import build_gold_market_macro_daily

mkt_df = spark.table("enterprise.silver_market.crypto_ohlc_1d")
fx_df = spark.table("enterprise.silver_ref.ecb_fx_daily")
fed_df = spark.table("enterprise.silver_ref.fred_series_daily")

# Build the table using the extracted function with a 5-day safe TTL limit
gold_macro_df = build_gold_market_macro_daily(mkt_df, fx_df, fed_df, max_fill_days=5)

gold_macro_df.write.mode("overwrite").format("delta").partitionBy("trade_date").saveAsTable("enterprise.gold_ref.market_macro_daily")

display(gold_macro_df.limit(10))


In [0]:
SELECT * FROM enterprise.gold_ref.market_macro_daily 
ORDER BY trade_date DESC 
LIMIT 100;

In [0]:
-- 查找 DFF 利率发生变化的日期
WITH lagged AS (
  SELECT 
    obs_date, 
    value, 
    LAG(value) OVER (ORDER BY obs_date) as prev_value 
  FROM enterprise.silver_ref.fred_series_daily
  WHERE series_id = 'DFF'
)
SELECT * FROM lagged 
WHERE value <> prev_value
ORDER BY obs_date DESC;

In [0]:
-- 从 Silver 表中删除非 DFF 的数据
DELETE FROM enterprise.silver_ref.fred_series_daily 
WHERE series_id != 'DFF';